# Voronoi and Proportional Cartogram Performance

This page shows the results of benchmarking the Voronoi cartogram and the proportional cartogram (shrinking and splitting). Benchmarks were run on two datasets (Albers equal-area projection, metres):

- **US states**: 48 contiguous US states plus the District of Columbia (49 regions, about 34,500 vertices), with Census population data.
- **Congressional districts**: 432 contiguous US congressional districts (2020 Census, geometries simplified at 1 km tolerance, about 92,000 vertices).

Each case is run for several rounds after one warm-up round. Times are the mean over the measured rounds; error bars show one standard deviation. Results are hardware-dependent and only comparable within one benchmark run.

In [ ]:
import json
import os

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import Markdown

_here = os.getcwd()
_json_path = next(
    (
        p
        for p in (
            os.path.join(_here, "benchmark_results.json"),
            os.path.join(_here, "docs", "performance", "benchmark_results.json"),
        )
        if os.path.exists(p)
    ),
    None,
)

_rows = []
if _json_path is not None and os.path.getsize(_json_path) > 0:
    with open(_json_path) as f:
        _data = json.load(f)
    for b in _data["benchmarks"]:
        extra = b.get("extra_info") or {}
        _rows.append({
            "name": b["name"],
            "config": b["name"].partition("[")[2].rstrip("]"),
            "n_regions": extra.get("n_regions"),
            "mean_s": b["stats"]["mean"],
            "stddev_s": b["stats"]["stddev"],
            "rounds": b["stats"]["rounds"],
        })
df_all = pd.DataFrame(_rows, columns=["name", "config", "n_regions", "mean_s", "stddev_s", "rounds"])
_DATASETS = {"us_states_gdf": "US States", "us_congressional_districts_gdf": "Congressional Districts"}


def _runtime_table(prefix, variant_col):
    """Rows of one benchmark test, with the dataset and variant taken from its id."""
    df = df_all[df_all["name"].str.startswith(prefix)].copy()
    df["dataset"] = df["config"].map(lambda c: _DATASETS.get(c.split("-")[0], c.split("-")[0]))
    if variant_col:
        df[variant_col] = df["config"].map(lambda c: c.split("-", 1)[1])
    df["n_regions"] = df["n_regions"].astype(int)
    return df.reset_index(drop=True)


def _markdown_table(df, columns, headers):
    md = "| " + " | ".join(headers) + " |\n"
    md += "|" + "|".join(" --- " for _ in headers) + "|\n"
    for _, row in df[columns].astype(object).iterrows():
        cells = [f"{v:.3f}" if isinstance(v, float) else str(v) for v in row]
        md += "| " + " | ".join(cells) + " |\n"
    return Markdown(md)

## Voronoi cartogram

This benchmark times `create_voronoi_cartogram` with its default settings: the raster backend (resolution 300), at most 30 Lloyd iterations, and population as the weight of each region. It measures the complete call, including the preparation of the outer boundary and the validation of the result.

The runtime grows roughly in proportion to the number of regions. The congressional districts have about nine times as many regions as the states and take about seven times as long.

In [ ]:
df_v = _runtime_table("test_voronoi_runtime", None)
display(
    _markdown_table(
        df_v, ["dataset", "n_regions", "mean_s", "stddev_s"], ["dataset", "regions", "mean (s)", "std dev (s)"]
    )
)

In [ ]:
_, ax = plt.subplots(figsize=(4, 4))
sns.barplot(data=df_v, x="dataset", y="mean_s", ax=ax, color="steelblue")
ax.errorbar(range(len(df_v)), df_v["mean_s"], yerr=df_v["stddev_s"], fmt="none", color="black", capsize=4)
ax.set_xlabel("")
ax.set_ylabel("mean runtime (s)")
ax.tick_params(axis="x", rotation=15)
plt.show()

## Proportional cartogram

This benchmark times `partition_geometries`, which divides every region into parts with given area fractions. Each region is divided into two parts, using its population relative to the largest population as the fraction. Two methods are compared:

- `shrink` creates a concentric core and a surrounding shell. It relies on repeated buffering of the region outline.
- `split` cuts the region with straight lines.

`split` is much faster than `shrink` on both datasets, and its runtime grows with the number of regions. The runtime of `shrink` does not: the states take longer than the districts although they are fewer, because each state is a larger and more detailed polygon (about 700 vertices per region, against about 210 for the simplified districts), which makes every buffering step more expensive.

In [ ]:
df_p = _runtime_table("test_partition_runtime", "method")
display(
    _markdown_table(
        df_p,
        ["dataset", "method", "n_regions", "mean_s", "stddev_s"],
        ["dataset", "method", "regions", "mean (s)", "std dev (s)"],
    )
)

In [ ]:
_, ax = plt.subplots(figsize=(5, 4))
sns.barplot(data=df_p, x="dataset", y="mean_s", hue="method", ax=ax)
ax.set_xlabel("")
ax.set_ylabel("mean runtime (s)")
plt.show()